# Project 1: Implementing Decision Trees


## Setup

### Importing libraries and the decision tree


In [1]:
#imports

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from decision_tree import DecisionTree, entropy, gini
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier

## Section 1 - Testing the decision tree implementation


### Test entropy and Gini on known label distributions


In [2]:
# All examples belong to one class: no impurity.
pure_labels = np.array([0, 0, 0, 0])

assert np.isclose(entropy(pure_labels), 0.0)
assert np.isclose(gini(pure_labels), 0.0)

# Two equally common classes.
balanced_labels = np.array([0, 0, 1, 1])

assert np.isclose(entropy(balanced_labels), 1.0)
assert np.isclose(gini(balanced_labels), 0.5)

# Three equally common classes.
three_classes = np.array([0, 1, 2])

assert np.isclose(entropy(three_classes), np.log2(3))
assert np.isclose(gini(three_classes), 2 / 3)

print("Impurity tests passed.")

Impurity tests passed.


### Test training and prediction on a small dataset


In [3]:
# Four examples, each with one feature.
toy_X = np.array([[1], [2], [8], [9]])
toy_y = np.array([0, 0, 1, 1])

# New examples whose expected predictions we can work out.
new_X = np.array([[3], [5], [7]])
expected = np.array([0, 0, 1])

for criterion in ["entropy", "gini"]:
    tree = DecisionTree(criterion=criterion)
    tree.fit(toy_X, toy_y)

    actual = tree.predict(new_X)

    np.testing.assert_array_equal(actual, expected)
    print(f"Training and prediction test passed: {criterion}")

Training and prediction test passed: entropy
Training and prediction test passed: gini


## Section 2 - Model selection and evaluation


### Loading customer data and separate features from churn labels


In [4]:
# Code given in instructions
df = np.genfromtxt("data/churn-data.csv", delimiter=",", dtype=float, names=True)
feature_names = list(df.dtype.names[:-1])
target_name = df.dtype.names[-1]
X = np.array([df[feature] for feature in feature_names]).T
y = df[target_name].astype(int)
print(f"Feature columns names: {feature_names}")
print(f"Target column name: {target_name}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Feature columns names: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Target column name: Churn
X shape: (7032, 19)
y shape: (7032,)


### Spliting data into training, validation, and test sets (60/20/20)


In [5]:
seed = 67

X_val_test, X_test, y_val_test, y_test = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_val_test, y_val_test,
    test_size=0.25, random_state=seed, stratify=y_val_test
)

# We chose 60/20/20 instead of 70/15/15 because larger validation and test sets
# make performance estimates less sensitive to which customers end up in each set.

### Training an initial tree and measure validation performance


In [ ]:
# Create a tree and train it.
model = DecisionTree() #max = none, entropy
model.fit(X_train, y_train)

# Predict labels for the validation customers.
val_predictions = model.predict(X_val)

# Compare predictions with the actual labels.
val_accuracy = accuracy_score(y_val, val_predictions)
val_f1 = f1_score(
    y_val, val_predictions, pos_label=1, zero_division=0
)

print(f"Validation accuracy test: {val_accuracy:.3f}")
print(f"Validation F1 test:       {val_f1:.3f}")

Validation accuracy test: 0.732
Validation F1 test:       0.501


### 2.1 - Choosing settings for our decision tree


In [7]:
results = []
best_f1 = -1
best_settings = None

# Testing both criterions and a set of different max depths
for criterion in ["entropy", "gini"]:
    for depth in [1, 2, 3, 5, 10, None]:

        candidate = DecisionTree(
            criterion=criterion,
            maxDepth=depth
        )
        candidate.fit(X_train, y_train)

        predictions = candidate.predict(X_val)

        accuracy = accuracy_score(y_val, predictions)
        f1 = f1_score(
            y_val, predictions, pos_label=1, zero_division=0
        )

        # Saving test results for every tree
        results.append({
            "criterion": criterion,
            "depth": "unlimited" if depth is None else depth,
            "accuracy": accuracy,
            "F1": f1
        })

        # Comparing last best tree-settings to current tree settings.
        # Setting best to current if current is better than best.
        if f1 > best_f1:
            best_f1 = f1
            best_settings = {
                "criterion": criterion,
                "maxDepth": depth
            }

results_table = pd.DataFrame(results)
display(results_table.sort_values("F1", ascending=False))

print("Best settings:", best_settings)
print(f"Best validation F1: {best_f1:.3f}")

,criterion,depth,accuracy,F1
9,gini,5,0.781805,0.601816
3,entropy,5,0.780384,0.593955
7,gini,2,0.732054,0.592432
1,entropy,2,0.732054,0.592432
8,gini,3,0.774698,0.549075
2,entropy,3,0.774698,0.549075
4,entropy,10,0.754087,0.539894
10,gini,10,0.746269,0.529644
11,gini,unlimited,0.725657,0.502577
5,entropy,unlimited,0.732054,0.500662


Best settings: {'criterion': 'gini', 'maxDepth': 5}
Best validation F1: 0.602


F1 accuracy is 0 when max depth is set to 1 becuase the tree predicts churn for every customer. Making it the worst setting.

### 2.2 - Chooseing settings for the sklearn decision tree


In [ ]:
sklearn_results = []
sklearn_best_f1 = -1
sklearn_best_settings = None

# Using the same function to test for best settings for sklearn decision tree as ours.
for criterion in ["entropy", "gini"]:
    for depth in [1, 2, 3, 5, 10, None]:
        candidate = DecisionTreeClassifier(
            criterion=criterion,
            max_depth=depth,
            random_state=seed
        )
        candidate.fit(X_train, y_train)

        predictions = candidate.predict(X_val)

        accuracy = accuracy_score(y_val, predictions)
        score = f1_score(
            y_val, predictions, pos_label=1, zero_division=0
        )

        sklearn_results.append({
            "criterion": criterion,
            "depth": "unlimited" if depth is None else depth,
            "accuracy": accuracy,
            "F1": score
        })

        if score > sklearn_best_f1:
            sklearn_best_f1 = score
            sklearn_best_settings = {
                "criterion": criterion,
                "max_depth": depth
            }

display(
    pd.DataFrame(sklearn_results)
    .sort_values("F1", ascending=False)
)

print("Best sklearn settings:", sklearn_best_settings)
print(f"Best sklearn validation F1: {sklearn_best_f1:.3f}")

,criterion,depth,accuracy,F1
1,entropy,2,0.732054,0.592432
7,gini,2,0.732054,0.592432
9,gini,5,0.796020,0.552262
3,entropy,5,0.791045,0.519608
10,gini,10,0.759773,0.508721
4,entropy,10,0.751244,0.501425
5,entropy,unlimited,0.726368,0.498044
11,gini,unlimited,0.718550,0.484375
8,gini,3,0.778252,0.417910
2,entropy,3,0.778252,0.417910


Best sklearn settings: {'criterion': 'entropy', 'max_depth': 2}
Best sklearn validation F1: 0.592


### Refiting both selected models and compare test performance


In [ ]:
final_custom = DecisionTree(
    criterion=best_settings["criterion"],
    maxDepth=best_settings["maxDepth"]
)

final_sklearn = DecisionTreeClassifier(
    criterion=sklearn_best_settings["criterion"],
    max_depth=sklearn_best_settings["max_depth"],
    random_state=seed
)

# Refit using the combined training and validation data.
final_custom.fit(X_val_test, y_val_test)
final_sklearn.fit(X_val_test, y_val_test)

# Evaluate the selected models on the held-out test set.
test_results = []

for name, fitted_model in [
    ("Our tree", final_custom),
    ("Sklearn tree", final_sklearn)
]:
    predictions = fitted_model.predict(X_test)

    test_results.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, predictions),
        "test_F1": f1_score(
            y_test, predictions, pos_label=1, zero_division=0
        )
    })

display(pd.DataFrame(test_results))

,model,test_accuracy,test_F1
0,Our tree,0.798152,0.573574
1,Sklearn tree,0.737029,0.579545
